# GOAL: Build linear regression model

In [1]:
#import statements

#import everything!
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, datetime, timedelta
from itertools import product
from copy import deepcopy
from PreRun import PreRun, PostRun
from sklearn.metrics import mean_absolute_error, mean_squared_error
#from statsmodels.tsa.statespace.sarimax import SARIMAX
import random
from sklearn.linear_model import LinearRegression


#path to data
read_path = "../../../../data_ds_project/parquet_cleaned_energy"
#systems
good_systems_list = [4, 10, 33, 36, 50, 51, 1199, 1204, 1283, 1284, 1289, 1332, 4902, 4903]
reader_types = ["meter", "inverter", None]
#systems_cleaned
systems_cleaned = pd.read_csv("../../../data/core/systems_cleaned.csv")


## Run with standard hour-cyclic

In [12]:
system_reader_pairs = [(4,None),(10,None), (33, None), (50,None), (51,None),(1283,'inverter'),(1283,'meter')]
#system_reader_pairs = [(1283,'inverter'),(1283,'meter')]


for pair in system_reader_pairs:
    system_id = pair[0]
    reader_type = pair[1]

    prerun = PreRun(system_id = system_id, meter_or_inverter = reader_type, path=read_path, systems_cleaned=systems_cleaned)
    prerun.load_data()
        
    prerun.fill_missing_hours()
    prerun.add_energy_features_only(daily_lags = 2, remove_daily_lags_nans=True, include_last_year = True, remove_last_year_nans=True, include_day_of_year_cyclic=True, include_hour_cyclic=True)

    prerun.add_weather_features_only()

    prerun.amended_data = prerun.amended_data.dropna()

    prerun.good_end_days_naive(1)
    prerun.tts_of_data_using_end_days()

    
    all_errors = []

    linReg = LinearRegression()
    system_recorded_max = prerun.data['energy'].max()
    first_debug = True
    for end_date in prerun.train_dates['date']:
        #print(type(date))
        data = prerun.data_until_ho_day(end_date)
        # end-date is the pd.Timestamp for midnight on the target day
        data_train = data[data['time'] < end_date - pd.Timedelta(days=1)]
        X_train = data_train.drop(columns=['time', 'energy'])
        y_train = data_train['energy']
        data_ho = data[(data['time'] >= end_date)
                       & (data['time'] < end_date + pd.Timedelta(days=1))]
        X_ho = data_ho.drop(columns=['time', 'energy'])
        y_ho = data_ho['energy']
        linReg.fit(X_train, y_train)

        y_pred = linReg.predict(X_ho)
        #make sure value between 0 and highest observed max
        y_pred = np.clip(y_pred, 0, system_recorded_max)
        #make sure that if it's not sunlight hours and irradiance = 0, then energy = 0
        darkness_mask = (~((X_ho['proportion_daytime']==0) & (X_ho['global_tilted_irradiance']==0))).astype(int) # = 0 when both sublight prop and irr are 0
        y_pred = y_pred*np.array(darkness_mask)
        error = PostRun.custom_error(y_ho, y_pred)
        all_errors.append(error)

    #save errors
    errors = pd.DataFrame(all_errors, columns = ['error'])
    errors.to_csv(f'linreg_errors/{system_id}_{reader_type}_linreg_errors.csv', index=False)



look at error distributions

In [3]:
#compare hyperparameters
#System 4
print('System 4, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/4_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 4, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 10, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/10_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 10, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 33, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/33_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 33, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 50, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/50_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 50, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()


print('System 51, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/51_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 51, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()



print('System 1283, Inverter, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/1283_inverter_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 1283, meter_or_inverter = 'inverter', path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 1283, Meter, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/1283_meter_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 1283, meter_or_inverter = 'meter', path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    



System 4, None, LinReg with streak = 1
recorded system max: 1.0241429033333334
  Hyperparameters: error
     mean: 0.024986753462464886, median: 0.0132771630809813, min: 0.0003305805778583, max: 0.3309694217968633, std: 0.03303769885654674

System 10, None, LinReg with streak = 1
recorded system max: 1.185825
  Hyperparameters: error
     mean: 0.03374331980846899, median: 0.0188945463773065, min: 0.0005048962623446, max: 0.3703001539990447, std: 0.043212403615684084

System 33, None, LinReg with streak = 1
recorded system max: 2.4044553333333334
  Hyperparameters: error
     mean: 0.13596942224687253, median: 0.0926083195049428, min: 0.0035260996695669, max: 1.8811362920707464, std: 0.14864986953604548

System 50, None, LinReg with streak = 1
recorded system max: 7.072975
  Hyperparameters: error
     mean: 1.1989274401962546, median: 0.6437368436968451, min: 0.0193212712706078, max: 14.255237431506282, std: 1.607186302249764

System 51, None, LinReg with streak = 1
recorded system ma

#### linreg vs sarimax with best hyperparam. ALL data.

System 4: unknown (linreg has better mean, but worse max)

System 10: linreg

System 33: unknown (sarimax has better max; linreg has better everything else)

System 50: linreg (by a lot)

System 51: linreg (by a lot)

System 1283, inverter: unknown (linreg has WAY better mean, but WAY worse max)

System 1283, meter: linreg (by a lot)



note that linreg (and naive baseline) should both get better the later you go. So I'll run the above with only the second half of linreg, then redo the comparisons.

In [2]:
#compare hyperparameters
#System 4
print('System 4, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/4_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 4, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 10, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/10_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 10, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 33, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/33_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 33, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 50, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/50_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 50, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()


print('System 51, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/51_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 51, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()



print('System 1283, Inverter, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/1283_inverter_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 1283, meter_or_inverter = 'inverter', path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 1283, Meter, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/1283_meter_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 1283, meter_or_inverter = 'meter', path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    



System 4, None, LinReg with streak = 1
recorded system max: 1.0241429033333334
  Hyperparameters: error
     mean: 0.020717946645913532, median: 0.01155390837571635, min: 0.0003305805778583, max: 0.1813287859143232, std: 0.025985943303680953

System 10, None, LinReg with streak = 1
recorded system max: 1.185825
  Hyperparameters: error
     mean: 0.028605320785223518, median: 0.015958699963558698, min: 0.0005048962623446, max: 0.2222956773802005, std: 0.033697664066014356

System 33, None, LinReg with streak = 1
recorded system max: 2.4044553333333334
  Hyperparameters: error
     mean: 0.11906243314687809, median: 0.0822668759357359, min: 0.0035260996695669, max: 1.0606166056649773, std: 0.12784487839920006

System 50, None, LinReg with streak = 1
recorded system max: 7.072975
  Hyperparameters: error
     mean: 0.8995753637319086, median: 0.5508698140949955, min: 0.0193212712706078, max: 11.54371492480927, std: 1.0856676778007088

System 51, None, LinReg with streak = 1
recorded syst

#### linreg vs sarimax with best hyperparam. Only second half of linreg.

System 4: linreg (new!)

System 10: linreg

System 33: unknown (sarimax has better max; linreg has better everything else. Same max as before.)

System 50: linreg (by a lot)

System 51: linreg (by a lot)

System 1283, inverter: linreg (new!)

System 1283, meter: linreg (by a lot)


2nd half linreg is better than 2nd half naive across the board.



### With Fourier

Now look at errors of linreg with more daily/hourly fourier terms (3)

In [10]:
system_reader_pairs = [(4,None),(10,None), (33, None), (50,None), (51,None),(1283,'inverter'),(1283,'meter')]
#system_reader_pairs = [(1283,'inverter'),(1283,'meter')]


for pair in system_reader_pairs:
    system_id = pair[0]
    reader_type = pair[1]

    prerun = PreRun(system_id = system_id, meter_or_inverter = reader_type, path=read_path, systems_cleaned=systems_cleaned)
    prerun.load_data()
        
    prerun.fill_missing_hours()
    prerun.add_energy_features_only(daily_lags = 2, remove_daily_lags_nans=True, include_last_year = True, remove_last_year_nans=True, include_day_of_year_cyclic=True, highest_fourier_term_hour = 3)

    prerun.add_weather_features_only()

    prerun.amended_data = prerun.amended_data.dropna()

    prerun.good_end_days_naive(1)
    prerun.tts_of_data_using_end_days()

    
    all_errors = []

    linReg = LinearRegression()
    system_recorded_max = prerun.data['energy'].max()
    first_debug = True
    for end_date in prerun.train_dates['date']:
        #print(type(date))
        data = prerun.data_until_ho_day(end_date)
        # end-date is the pd.Timestamp for midnight on the target day
        data_train = data[data['time'] < end_date - pd.Timedelta(days=1)]
        X_train = data_train.drop(columns=['time', 'energy'])
        y_train = data_train['energy']
        data_ho = data[(data['time'] >= end_date)
                       & (data['time'] < end_date + pd.Timedelta(days=1))]
        X_ho = data_ho.drop(columns=['time', 'energy'])
        y_ho = data_ho['energy']
        linReg.fit(X_train, y_train)

        y_pred = linReg.predict(X_ho)
        #make sure value between 0 and highest observed max
        y_pred = np.clip(y_pred, 0, system_recorded_max)
        #make sure that if it's not sunlight hours and irradiance = 0, then energy = 0
        darkness_mask = (~((X_ho['proportion_daytime']==0) & (X_ho['global_tilted_irradiance']==0))).astype(int) # = 0 when both sublight prop and irr are 0
        y_pred = y_pred*np.array(darkness_mask)
        error = PostRun.custom_error(y_ho, y_pred)
        all_errors.append(error)

    #save errors
    errors = pd.DataFrame(all_errors, columns = ['error'])
    errors.to_csv(f'linreg_errors/{system_id}_{reader_type}_linreg_errors_fourier.csv', index=False)



In [ ]:
#compare hyperparameters
for pair in system_reader_pairs:
    system_id = pair[0]
    reader_type = pair[1]
    print(f'System {system_id}, {reader_type}, LinReg with streak = 1')
    errors = pd.read_csv(f'linreg_errors/{system_id}_{reader_type}_linreg_errors_fourier.csv')
    hyperparams = errors.columns
    prerun = PreRun(system_id = system_id, meter_or_inverter = reader_type, path=read_path, systems_cleaned=systems_cleaned)
    prerun.load_data()
    system_recorded_max = prerun.data['energy'].max()
    print(f'recorded system max: {system_recorded_max}')

    for hp in hyperparams:
        err = errors[hp]
        print(f'  Hyperparameters: {hp}')
        print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    print('\n')

System 4, None, LinReg with streak = 1
recorded system max: 1.0241429033333334
  Hyperparameters: error
     mean: 0.025008424680691, median: 0.012913323884813, min: 0.0003356992495231, max: 0.332472842477465, std: 0.03335059223862556


System 10, None, LinReg with streak = 1
recorded system max: 1.185825
  Hyperparameters: error
     mean: 0.03371136604269641, median: 0.01877256202082695, min: 0.0003474554895851, max: 0.369017178331716, std: 0.043173814486928176


System 33, None, LinReg with streak = 1
recorded system max: 2.4044553333333334
  Hyperparameters: error
     mean: 0.1351332875838896, median: 0.0921196558939404, min: 0.0034439993963383, max: 1.8951278626320431, std: 0.14971785080330233


System 50, None, LinReg with streak = 1
recorded system max: 7.072975
  Hyperparameters: error
     mean: 1.1903000136406545, median: 0.6364931311472842, min: 0.0159851824228365, max: 14.219866457489612, std: 1.608456997756673


System 51, None, LinReg with streak = 1
recorded system max:

and with only the second half of the outputs!

In [6]:
#compare hyperparameters
#System 4
print('System 4, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/4_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 4, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 10, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/10_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 10, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 33, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/33_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 33, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 50, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/50_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 50, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()


print('System 51, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/51_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 51, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()



print('System 1283, Inverter, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/1283_inverter_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 1283, meter_or_inverter = 'inverter', path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 1283, Meter, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/1283_meter_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 1283, meter_or_inverter = 'meter', path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    



System 4, None, LinReg with streak = 1
recorded system max: 1.0241429033333334
  Hyperparameters: error
     mean: 0.020692753835334924, median: 0.011415429908465651, min: 0.0003356992495231, max: 0.1812100375886306, std: 0.026277375017625056

System 10, None, LinReg with streak = 1
recorded system max: 1.185825
  Hyperparameters: error
     mean: 0.028586403198390866, median: 0.01574169185158445, min: 0.0003474554895851, max: 0.2175067382236179, std: 0.03390890723103508

System 33, None, LinReg with streak = 1
recorded system max: 2.4044553333333334
  Hyperparameters: error
     mean: 0.11807308323104802, median: 0.0815548683047693, min: 0.0034439993963383, max: 1.0645767379611533, std: 0.1280652264353099

System 50, None, LinReg with streak = 1
recorded system max: 7.072975
  Hyperparameters: error
     mean: 0.8849296258458285, median: 0.5353718121850002, min: 0.0159851824228365, max: 11.612355264681732, std: 1.0719069707085287

System 51, None, LinReg with streak = 1
recorded syste